## 初始化库，进行数据清洗：缺失值 + 异常值😆

In [3]:
# ===================== 模块1：导入所需工具库 =====================
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import numpy as np

# 设置中文显示，避免画图乱码
plt.rcParams["font.family"] = "Microsoft YaHei"
plt.rcParams["axes.unicode_minus"] = False

# ===================== 模块2：创建输出文件夹（自动创建，防止路径报错） =====================
os.makedirs("../data/processed", exist_ok=True)   # 清洗后数据存放路径
os.makedirs("../data/output-img", exist_ok=True)  # 可视化图片存放路径

# ===================== 模块3：加载原始数据集 =====================
df = pd.read_csv("../data/raw/cs-training.csv", index_col=0)

# 输出基础信息
print("="*60)
print("✅ 原始数据加载成功！")
print(f"数据形状：{df.shape[0]} 行，{df.shape[1]} 列")
print("="*60)

✅ 原始数据加载成功！
数据形状：150000 行，11 列


## 缺失值填充

根据EDA可知，月收入和家属收入有数据缺失
- 月收入：中位数填充
- 家属人数：0填充

In [4]:
# ===================== 模块4：缺失值填充 =====================
print("\n🔧 开始执行【缺失值填充】")
print("="*40)

# 1. 查看清洗前缺失值情况
print("📌 填充前缺失值统计：")
missing_before = pd.DataFrame({
    "缺失数量": df.isnull().sum(),
    "缺失比例(%)": (df.isnull().sum() / len(df) * 100).round(2)
}).loc[["MonthlyIncome", "NumberOfDependents"]]
print(missing_before)

# 2. 执行填充（金融数据标准规则）
df["MonthlyIncome"] = df["MonthlyIncome"].fillna(df["MonthlyIncome"].median())       # 月收入：中位数填充
df["NumberOfDependents"] = df["NumberOfDependents"].fillna(0)                       # 家属人数：0填充

# 3. 验证填充结果
print("\n✅ 缺失值填充完成！")
print("📌 填充后缺失值统计：")
missing_after = pd.DataFrame({
    "缺失数量": df.isnull().sum(),
    "缺失比例(%)": (df.isnull().sum() / len(df) * 100).round(2)
}).loc[["MonthlyIncome", "NumberOfDependents"]]
print(missing_after)
print("="*60)


🔧 开始执行【缺失值填充】
📌 填充前缺失值统计：
                     缺失数量  缺失比例(%)
MonthlyIncome       29731    19.82
NumberOfDependents   3924     2.62

✅ 缺失值填充完成！
📌 填充后缺失值统计：
                    缺失数量  缺失比例(%)
MonthlyIncome          0      0.0
NumberOfDependents     0      0.0


## 基于信用评分卡业务规则处理异常值
针对 DebtRatio存在过多异常大的值（异常20.87%）
 - 如此巨大的数值极不合理，可能由于收入接近0或数据录入错误导致

针对 逾期天数（30-59，60-89，大于90）
  - 出现割裂的96和98的数值，与其他值差距极大
  - 考虑到98常作为缺失值编码（如98表示未知）

针对MonthlyIncome（月收入）字段中存在的 0 值
 - 业务层面：信用卡授信要求用户具备稳定还款能力，月收入为 0 的样本不具备真实业务代表性；
 - 数据层面：0 值会使收入分布极端偏斜，会干扰模型对还款能力特征的学习；
 - 处理方案：采用中位数替换策略

针对 NumberOfDependents（家属数量）
- 出现一个断层的20家属，13个家属：同样极不合理，并且都只有1个，建议处理。
- 10个家属：理论上可能存在。考虑到只有5个样本，可以保留，但需警惕其对模型的潜在影响。
- NA值：缺失值占比约（3924/总样本数），需要妥善处理。


针对 RevolvingUtilizationOfUnsecuredLines（循环额度使用率）
- 通常应在0~1之间（超过1表示超额使用，但一般不会太大）
- IQR正常范围为[-0.76, 1.35]，实际最大值50,708，异常比例0.51%。显然，50,708是极端错误值，可能由于分母过小或数据录入错误

针对 Age 和 NumberRealEstateLoansOrLines（房产数量）
- 成年岁数数量，大于18
- 出现54的过大值，可能删除

In [5]:
# ===================== 模块5：异常值处理（重构版·分步计数） =====================
print("\n🔧 开始执行【分步骤异常值处理】")
print("="*60)

# 保存原始数据总量，用于后续统计
original_count = len(df)
print(f"📊 处理前总样本数：{original_count} 条")

# -----------------------------------------------------------------------------
# 【处理1】删除三类逾期次数字段中 =96 / =98 的异常样本
# -----------------------------------------------------------------------------
due_cols = [
    "NumberOfTime30-59DaysPastDueNotWorse",
    "NumberOfTime60-89DaysPastDueNotWorse",
    "NumberOfTimes90DaysLate"
]
# 标记：任意逾期字段为96或98的行
mask_due = False
for col in due_cols:
    mask_due |= (df[col] == 96) | (df[col] == 98)

delete_due_count = mask_due.sum()  # 统计要删除的数量
df = df[~mask_due]  # 删除异常行
print(f"\n1. 逾期异常值处理：删除值为96/98的样本 ✅")
print(f"   删除数量：{delete_due_count} 条")
print(f"   当前剩余样本：{len(df)} 条")

# -----------------------------------------------------------------------------
# 【处理2】删除年龄 < 18 岁的无效样本（未成年人无信贷资格）
# -----------------------------------------------------------------------------
mask_age = df["age"] < 18
delete_age_count = mask_age.sum()
df = df[~mask_age]
print(f"\n2. 年龄异常值处理：删除<18岁的无效样本 ✅")
print(f"   删除数量：{delete_age_count} 条")
print(f"   当前剩余样本：{len(df)} 条")

# -----------------------------------------------------------------------------
# 【处理3】月收入处理：0值 + 缺失值(NA) 统一填充为中位数
# -----------------------------------------------------------------------------
# 统计需要填充的数量
income_zero_count = (df["MonthlyIncome"] == 0).sum()
income_na_count = df["MonthlyIncome"].isnull().sum()
income_fill_total = income_zero_count + income_na_count
# 填充中位数
income_median = df["MonthlyIncome"].median()
df["MonthlyIncome"] = df["MonthlyIncome"].replace(0, income_median)
df["MonthlyIncome"] = df["MonthlyIncome"].fillna(income_median)

print(f"\n3. 月收入缺失/异常值处理：填充中位数 ✅")
print(f"   填充0值数量：{income_zero_count} 条")
print(f"   填充缺失值(NA)数量：{income_na_count} 条")
print(f"   总填充数量：{income_fill_total} 条")

# -----------------------------------------------------------------------------
# 最终总结
# -----------------------------------------------------------------------------
print("\n" + "="*60)
print("✅ 【全部异常值处理完成】最终统计")
print(f"原始样本数：{original_count} 条")
print(f"最终样本数：{len(df)} 条")
print(f"累计删除异常样本：{original_count - len(df)} 条")
print(f"累计填充月收入异常/缺失：{income_fill_total} 条")
print("="*60)

# 输出处理后核心特征统计
print("\n📌 处理后核心特征统计：")
outlier_cols = [ "age", "DebtRatio", "MonthlyIncome"]
print(df[outlier_cols].describe().round(2))
print("="*60)


🔧 开始执行【分步骤异常值处理】
📊 处理前总样本数：150000 条

1. 逾期异常值处理：删除值为96/98的样本 ✅
   删除数量：269 条
   当前剩余样本：149731 条

2. 年龄异常值处理：删除<18岁的无效样本 ✅
   删除数量：1 条
   当前剩余样本：149730 条

3. 月收入缺失/异常值处理：填充中位数 ✅
   填充0值数量：1627 条
   填充缺失值(NA)数量：0 条
   总填充数量：1627 条

✅ 【全部异常值处理完成】最终统计
原始样本数：150000 条
最终样本数：149730 条
累计删除异常样本：270 条
累计填充月收入异常/缺失：1627 条

📌 处理后核心特征统计：
             age  DebtRatio  MonthlyIncome
count  149730.00  149730.00      149730.00
mean       52.33     353.63        6481.78
std        14.75    2039.60       12884.04
min        21.00       0.00           1.00
25%        41.00       0.18        4000.00
50%        52.00       0.37        5400.00
75%        63.00       0.87        7400.00
max       109.00  329664.00     3008750.00


## 第二部分的异常数据处理
- 使用**截尾**（winsorize / clip）：保留样本，但把极端值压到一个合理范围。
    - 小于下限的值 → 强制改成下限，
    - 大于上限的值 → 强制改成上限
- 使用**log 转换**（log transform）对变量做数学变换，让分布更平滑。

虽然部分极端值可能反映真实风险，但为了模型稳定性与可解释性，采用截尾处理
- DebtRatio使用硬截尾（0-10），负债是收入的10倍，再多就没有使用的意义了
- 循环额度使用率使用硬截尾（0-25）

DNN训练需要注意：极端值，长尾，尺度差异


## 可视化 + 保存数据 + 清洗总结

In [ ]:
# ===================== 模块6：异常值处理结果可视化（毕设专用） =====================
plt.figure(figsize=(16, 10))
features = ["RevolvingUtilizationOfUnsecuredLines", "DebtRatio", "MonthlyIncome", "age"]
titles = ["循环额度使用率", "债务率", "月收入", "年龄"]
colors = ["#3498db", "#e67e22", "#9b59b6", "#2ecc71"]

for i, (col, title, color) in enumerate(zip(features, titles, colors), 1):
    plt.subplot(2, 2, i)
    sns.boxplot(y=df[col], color=color)
    plt.title(f"{title}（处理后）", fontweight="bold")

plt.tight_layout()
plt.savefig("../data/output-img/异常值处理后箱线图.png", dpi=300, bbox_inches="tight")
plt.show()

# ===================== 模块7：保存清洗后的数据 =====================
df.to_csv("../data/processed/cs-training-cleaned.csv", index=True)
print("\n💾 清洗后数据已保存：../data/processed/cs-training-cleaned.csv")

# ===================== 模块8：数据清洗总结报告（可直接复制进毕设） =====================
print("\n" + "="*60)
print("📋 数据清洗总结报告")
print("="*60)
print(f"1. 缺失值：完成月收入、家属人数缺失填充，数据无缺失；")
print(f"2. 异常值：基于业务规则修正极端值，数据符合金融建模规范；")
print(f"3. 输出结果：清洗后数据已保存，可直接用于后续数据聚合与标准化；")
print("="*60)